## (Optional) Initialize state from environment variable, for command line execution

In [1]:
import os
# Read fitness data from CSV files
main_dir = os.getenv("MAIN_DIR", "results/sampling")
# Save outputs here
output_dir = os.getenv("OUTPUT_DIR", "results/filtering")
os.makedirs(output_dir, exist_ok=True)
# Filtering threshold for sensitivity
MEAN_THRESHOLD = os.getenv("MEAN_THRESHOLD", 0.0)
ABS_STD_THRESHOLD = os.getenv("ABS_STD_THRESHOLD", 30.0)
NORM_STD_THRESHOLD = os.getenv("NORM_STD_THRESHOLD", 0.2)

In [5]:
main_dir

'../../results/sampling'

# Load Dataset

Dataset of simulations is readed from file system and inserted into an np array.

In [4]:
import numpy as np
import os
import csv
import matplotlib.pyplot as plt

dataset = []
i=0
broken_reads = 0
while True:
    file_name = os.path.join(main_dir, f"{i}_fitness_step.csv")
    if not os.path.exists(file_name):
        break
    context_ds = []
    with open(file_name, 'r') as csvfile:
        reader = csv.reader(csvfile)
        next(reader, None)  # Skip the header row
        for row in reader:
            if len(row) > 0:
                try:
                    context_ds.append([float(x) for x in row])
                except ValueError:
                    #print(f"Warning: Non-numeric data found in {file_name}, skipping row: {row}")
                    broken_reads += 1
                    context_ds.append([np.nan for x in row])
    if len(context_ds) == 0:
        print(f"Warning: {file_name} is empty, skipping.")
        i += 1
        continue
    dataset.append(context_ds)
    i += 1

dataset = np.array(dataset)

function_names = ["AliveCellsFitness","CircularFitness","CircularFitness_WT_DISTRIBUTION","SquaredFitness","SquaredFitness_WT_DISTRIBUTION","CircularFitness_2","CircularFitness_2_WT_DISTRIBUTION","SquaredFitness_2","SquaredFitness_2_WT_DISTRIBUTION"]


print(f"Dataset loaded: {dataset.shape}")
print(f"Function names: {function_names}")
print(f"Broken reads: {broken_reads}")

Dataset loaded: (0,)
Function names: ['AliveCellsFitness', 'CircularFitness', 'CircularFitness_WT_DISTRIBUTION', 'SquaredFitness', 'SquaredFitness_WT_DISTRIBUTION', 'CircularFitness_2', 'CircularFitness_2_WT_DISTRIBUTION', 'SquaredFitness_2', 'SquaredFitness_2_WT_DISTRIBUTION']
Broken reads: 0


# Clean dataset

Some linear fitness function might contain NaNs, which will break following analysis.
NaNs are replaced with column average: this way they don't artificially improve the measured standard deviation.

In [ ]:
number_of_nans = np.sum(np.isnan(dataset))
# Replace NaNs with the average of the respective function across all contexts
for i in range(dataset.shape[0]):
    for j in range(dataset.shape[2]):
        avg = np.nanmean(dataset[i,:,j])
        if (np.isnan(avg)):
            avg = 0.0
        dataset[i,np.isnan(dataset[i,:,j]),j] = avg

print(f"Dataset after NaN replacement: {dataset.shape}")

## Early visualization: PCA to visualize models

### First analysis plot considers the entire dataset: 
* Different fitness functions over same models seen as separate models.
* It's possible to identify 8 clear tracks: most likely, they show 8 of the 9 fitness functions used.

In [ ]:
flatten_dataset = dataset.reshape(-1, dataset.shape[1])
print(f"Flattened dataset shape: {flatten_dataset.shape}")
def pca(X, num_components=2):
    # Center the data
    X_centered = X - np.mean(X)
    # Compute covariance matrix
    covariance_matrix = np.cov(X_centered, rowvar=False)
    # Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
    # Sort eigenvalues and eigenvectors
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    # Select top components
    selected_eigenvectors = sorted_eigenvectors[:, :num_components]
    # Project data
    X_reduced = np.dot(X_centered, selected_eigenvectors)
    return X_reduced, sorted_eigenvalues[:num_components]


def plot_pca(components, labels=None):
    plt.figure(figsize=(10, 7))
    plt.scatter(components[:, 0], components[:, 1], alpha=0.5)
    plt.title('PCA of Models based on Fitness Functions')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.grid()
    plt.show()

components, _ = pca(flatten_dataset, num_components=2)
plot_pca(components)


## Variance analysis

### Display quartiles for variance across models

In [ ]:
for i in range(dataset.shape[2]):
    best_indices = np.argsort(variance_vector[:, i])[-5:]  # Indices of the 5 highest variances
    print(f"Function: {function_names[i]}, Top 5 variances: {", ".join([str(t) for t in variance_vector[best_indices, i]])}")
    #Plot quartiles
    quartiles = np.percentile(variance_vector[:, i], [25, 50, 75])
    plt.figure(figsize=(6, 4))
    plt.bar(['Q1', 'Q2', 'Q3'], quartiles, color='orange', edgecolor='black')
    plt.title(f"Quartiles of Variance for {function_names[i]}")
    plt.ylabel('Variance')
    plt.show()

### Same as before but instead of variance display the CV

In [ ]:
std_vector = np.std(dataset, axis=1)
std_vector_normalized = std_vector / np.mean(dataset, axis=1)  # Variance across contexts for each function
std_vector_normalized = np.nan_to_num(std_vector_normalized, nan=0.0)
print(std_vector_normalized.shape)

In [ ]:
for i in range(dataset.shape[2]):
    best_indices = np.argsort(std_vector_normalized[:, i])[-5:]  # Indices of the 5 highest variances
    print(f"Function: {function_names[i]}, Top 5 variances: {", ".join([str(t) for t in std_vector_normalized[best_indices, i]])}")
    #Plot quartiles
    quartiles = np.percentile(std_vector_normalized[:, i], [25, 50, 75])
    plt.figure(figsize=(6, 4))
    plt.bar(['Q1', 'Q2', 'Q3'], quartiles, color='orange', edgecolor='black')
    plt.title(f"Quartiles of Variance for {function_names[i]}")
    plt.ylabel('Variance')
    plt.show()

### Display the distribution of standard deviation across models: for each bin representing an interval of st. dev., plot the number of models with this standard deviation.

In [ ]:

for i in range(len(function_names)):
    plt.figure(figsize=(10, 6))
    plt.hist(std_vector[:, i], bins=100, alpha=0.7, color='blue', edgecolor='black')
    plt.title(f"Standard deviation distribution for {function_names[i]}")
    plt.xlabel('Variance')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

## Filter models:

Two types of filters are supported:

1. Filter by Mean: keep only models that exhibit mean >= MEAN_THRESHOLD.
2. Filter by variance and CV, using two conditions in OR (keep models that respect at least one):
    * Standard deviation >= ABS_STD_THRESHOLD.
    * CV >= NORM_STD_THRESHOLD.

The parameters are defined at the head of this Notebook:

*MEAN_THRESHOLD = 0.0*

*ABS_STD_THRESHOLD = 30.0*

*NORM_STD_THRESHOLD = 0.2*

#### Prepare variables to filter the dataset while preserving information on the origin of each model

In [ ]:
# Test- get rid of fitness functions != 0
#dataset = dataset[:,:,0]
dataset_by_f = [
    dataset[:,:,i:i+1] for i in range(dataset.shape[2])
]

In [ ]:
def filter_by_mean(threshold):
    mean_values = np.mean(flattened_ds[:, :], axis=1)
    mask = mean_values > threshold
    return mask

def filter_by_standard_deviation(threshold):
    std_vector = np.std(flattened_ds, axis=1)
    mask = std_vector > threshold
    return mask

def filter_by_standard_deviation_normalized(threshold):
    std_vector = np.std(flattened_ds, axis=1)
    std_vector_normalized = std_vector / np.mean(flattened_ds, axis=1)
    mask = std_vector_normalized > threshold
    return mask

filtered_indices_by_f = []
# Apply filters to each individual fitness function
for dataset in dataset_by_f:
    print("\nProcessing a single fitness function dataset")
    # Flatten the dataset to 2D: (num_models * num_fitness_functions, num_contexts)
    original_model_indexes = np.repeat(np.arange(dataset.shape[0]), dataset.shape[2])
    function_indexes = np.tile(np.arange(dataset.shape[2]), dataset.shape[0])
    flattened_ds = np.transpose(dataset, (0, 2, 1)).reshape(-1, dataset.shape[1])
    mean_mask = filter_by_mean(MEAN_THRESHOLD)
    std_mask = filter_by_standard_deviation(ABS_STD_THRESHOLD)
    std_norm_mask = filter_by_standard_deviation_normalized(NORM_STD_THRESHOLD)
    combined_mask = mean_mask & (std_mask | std_norm_mask)
    filtered_indices = np.where(combined_mask)[0]
    print(f"Number of models after filtering: {len(filtered_indices)} out of {flattened_ds.shape[0]}")
    print(f"The mean mask retains {np.sum(mean_mask)} models.")
    print(f"The absolute std mask retains {np.sum(std_mask)} models.")
    print(f"The normalized std mask retains {np.sum(std_norm_mask)} models.")
    print(f"The combined absolute and normalized std masks retains {np.sum(combined_mask)} models.")

    filtered_dataset = flattened_ds[filtered_indices, :]
    filtered_model_indexes = original_model_indexes[filtered_indices]
    filtered_function_indexes = function_indexes[filtered_indices]
    filtered_indices_by_f.append(filtered_indices)
    print(f"Filtered dataset shape: {filtered_dataset.shape}")

# Find models that satisfy all:
filtered_indices_by_f = [
    set(indices) for indices in filtered_indices_by_f
]
all_indices = set.intersection(*filtered_indices_by_f)
flattened_ds = np.transpose(dataset, (0, 2, 1)).reshape(-1, dataset.shape[1])
print(f"\nNumber of models satisfying all fitness functions: {len(all_indices)} out of {flattened_ds.shape[0]}")
filtered_dataset = flattened_ds[list(all_indices), :]
filtered_function_indexes = function_indexes[list(all_indices)]


In [ ]:
# Export filtered dataset to final_pool
all_indices = np.array(list(all_indices))
import csv
all_description = []
with open(os.path.join(main_dir, 'subjects.csv'), 'r') as f:
    row = csv.reader(f)
    for row in f:
        row = row.strip().split(',')
        family, model = row[8], row[9]
        all_description.append((family, model))
all_description = np.array(all_description)
final = all_description[all_indices]
print(all_indices.shape, all_description.shape, final.shape)
with open(os.path.join(output_dir, 'final_pool.csv'), 'w') as f:
    for entry in final:
        f.write(f"{entry[0]},{entry[1]}\n")

### Further analysis

In [ ]:
correlation_matrix = np.corrcoef(filtered_dataset)
# Method 1: Using matplotlib
plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.title(f"Correlation Matrix after Filtering - Function: {function_names[0]}")
plt.xlabel('Vector Index')
plt.ylabel('Vector Index')
plt.show()
# Max correlation
correlation_matrix_no_diagonal = correlation_matrix.copy()
np.fill_diagonal(correlation_matrix_no_diagonal, 0) # Ignore self-correlation
max_abs_correlation = np.max(np.abs(correlation_matrix_no_diagonal))
avg_abc_correlation = np.mean(np.abs(correlation_matrix_no_diagonal))
print(f"Avg abs correlation: {avg_abc_correlation}, max abs correlation: {max_abs_correlation}")

# Filter out models with correlation >= MAX_CORRELATION
# Implemented as a loop that removes one element at a time and recomputes the correlation matrix
# (not the most efficient implementation, but simple and the dataset is small enough)
num_removed = 0
while(True):
    correlation_matrix = np.corrcoef(filtered_dataset)
    np.fill_diagonal(correlation_matrix, 0) # Ignore self-correlation
    to_remove = np.where(np.abs(correlation_matrix) >= MAX_CORRELATION)
    if len(to_remove[0]) == 0:
        break
    # Remove one model. Prefer to remove one that is not of the first function (index 0)
    found = False
    for idx in range(len(to_remove[0])):
        if (filtered_function_indexes[to_remove[0][idx]] != 0):
            index_to_remove = to_remove[0][idx]
            found = True
            break
    if not found:
        index_to_remove = to_remove[0][0]
    filtered_dataset = np.delete(filtered_dataset, index_to_remove, axis=0)
    filtered_model_indexes = np.delete(filtered_model_indexes, index_to_remove, axis=0)
    filtered_function_indexes = np.delete(filtered_function_indexes, index_to_remove, axis=0)
    num_removed += 1
    if (num_removed % 100 == 0):
        print(f"Removed {num_removed} models so far, current shape: {filtered_dataset.shape}")



## Label models in filtered_dataset by the existence of outliers
Each model is scored by the number of outliers it exhibited and their strength.

In [ ]:
import numpy as np

outlier_strength_scores = []

for values in filtered_dataset:
    values = np.asarray(values)
    median = np.median(values)
    mad = np.median(np.abs(values - median))

    if mad == 0:
        outlier_strength_scores.append(0)
        continue

    modified_z = 0.6745 * (values - median) / mad
    abs_mz = np.abs(modified_z)

    threshold = 3.5
    outliers = abs_mz > threshold

    # Outlier strength = how far points exceed typical outlier threshold
    strength = np.sum(abs_mz[outliers] - threshold)
    outlier_strength_scores.append(strength)


print(f"Outlier strength scores:", "\n".join([f"\t{i}: {score}" for i, score in enumerate(outlier_strength_scores)]))
print(f"Max outlier strength score: {max(outlier_strength_scores)}")
outlier_strength_scores = np.array(outlier_strength_scores)
print(f"Average and quartiles of outlier strength scores:", np.mean(outlier_strength_scores), np.percentile(outlier_strength_scores, [25, 50, 75]))

# 

In [ ]:
# Select 10 models for meta-modelling:
final_dataset_only_fitness = final_dataset[filtered_function_indexes == 0, :]
filtered_model_indexes_only_fitness = filtered_model_indexes[filtered_function_indexes == 0]
outlier_scores_only_fitness = np.array(outlier_scores)[filtered_function_indexes == 0]

# Chose 5 models with highest outlier scores
top_outlier_indices = np.argsort(outlier_scores_only_fitness)[-5:]
selected_models = final_dataset_only_fitness[top_outlier_indices, :]
selected_model_indexes = filtered_model_indexes_only_fitness[top_outlier_indices]
print(f"Models selected by high outliers score shape: {selected_models.shape}")

models_not_selected =  final_dataset_only_fitness[~np.isin(np.arange(final_dataset_only_fitness.shape[0]), top_outlier_indices), :]
model_indexes_not_selected = filtered_model_indexes_only_fitness[~np.isin(np.arange(final_dataset_only_fitness.shape[0]), top_outlier_indices)]
# From the remaining models, choose 5 with highest variance
remaining_variances = np.var(models_not_selected, axis=1)
top_variance_indices = np.argsort(remaining_variances)[-5:]
selected_models_variance = models_not_selected[top_variance_indices, :]
selected_model_indexes_variance = model_indexes_not_selected[top_variance_indices]
print(f"Models selected by high variance shape: {selected_models_variance.shape}")

def print_model(model_string):
    m = model_string.split(',')
    # Model info
    model_family = m[-2]
    model_name = m[-1]
    # Initial positions info
    init_pos_type = m[1]
    init_pos_center = (m[2], m[3])
    init_pos_density = m[4]
    init_pos_cell_type = m[5]
    init_pos_mode = m[6]
    init_pos_length = m[7]
    print(f"Family: {model_family}; Model: {model_name.strip()}")
    print(f"\tInitial positions:\n\t\tType: {init_pos_type}\n\t\tCenter: {init_pos_center}\n\t\tDensity: {init_pos_density}\n\t\tCell type: {init_pos_cell_type}\n\t\tMode: {init_pos_mode}\n\t\tLength: {init_pos_length}\n\n")


# Print selected models
out_csv = open("final_pool.csv", "w")
print(f"Selected model for high outlier scores:")
with open("subjects.csv", "r") as f:
    lines = f.readlines()
    for idx in selected_model_indexes:
        if idx < len(lines):
            print_model(lines[idx])
        else:
            print(f"Model index {idx}: Not found in subjects.csv")
P
print(f"\nSelected model for high variance:")
with open("subjects.csv", "r") as f:
    lines = f.readlines()
    for idx in selected_model_indexes_variance:
        if idx < len(lines):
            print_model(lines[idx])
        else:
            print(f"Model index {idx}: Not found in subjects.csv")